# Notebook - Phase 6

## Triage of the 11 problematic tickers

In [1]:
import yaml
with open("/Users/hugo/quant-research/config/equity_universes.yaml", 'r') as f:
    config = yaml.safe_load(f)
universe = 'russell1000'

tickers = config[universe]
print(len(tickers))

1005


In [14]:
import pandas as pd
df = pd.read_csv("/Users/hugo/quant-research/config/russell1000_sectors.csv")
df.head()

,Company,Symbol,GICS Sector,GICS Sub-Industry
0,3M,MMM,Industrials,Industrial Conglomerates
1,A. O. Smith,AOS,Industrials,Building Products
2,AAON,AAON,Industrials,Building Products
3,Abbott Laboratories,ABT,Health Care,Health Care Equipment
4,AbbVie,ABBV,Health Care,Pharmaceuticals


In [15]:
# check if symbols match
df_tolist = df["Symbol"].to_list()
print(set(df_tolist)-set(tickers))
print(set(tickers)-set(df_tolist))

{'HEI.A', 'LEN.B', 'BF.A', 'UHAL.B', 'BF.B', 'CWEN.A', 'BRK.B'}
{'UHAL B', 'BF B', 'BRK B', 'LEN B', 'BF A', 'CWEN A', 'HEI A'}


In [16]:
def normalize(x):
    return x.strip().replace("."," ")

In [19]:
df["Symbol"] = df["Symbol"].apply(normalize)
print(set(df["Symbol"].tolist())-set(tickers))

set()


Creation of the `.csv` file where status/rename/industry/exchange/delist info live

First create a DataFrame and export to csv. Columns:

`conID | ticker | secIdType | secId | sec_type | primary_exchange | currency | gics_sector | status | delist_date | former_ticker | last_fetch`

In [29]:
import nest_asyncio; nest_asyncio.apply()
from src.execution.ib_connection import IBConnection
from ib_async import Stock
conn = IBConnection()
conn.connect()

print(f"Connected: {conn.is_connected()}")
print(f"Accounts: {conn.ib.managedAccounts()}")

cds = conn.ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

conn.disconnect()

open orders request timed out
completed orders request timed out


Connected: True
Accounts: ['U24347775']
1 265598 NASDAQ


In [24]:
import nest_asyncio; nest_asyncio.apply()      # needed for ib_async in Jupyter
from ib_async import Stock
from src.execution.ib_connection import IBConnection

with IBConnection(client_id=1) as ib:           # host/port from .env
    cds = ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
    print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

open orders request timed out
completed orders request timed out


1 265598 NASDAQ
